In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.inspection import permutation_importance

In [57]:
ERA5_DISPLAY_NAMES = {
    "2m_temperature":                           "t2m",
    "2m_dewpoint_temperature":                  "d2m",
    "10m_u_component_of_wind":                  "u10",
    "10m_v_component_of_wind":                  "v10",
    "surface_pressure":                         "sp",
    "total_cloud_cover":                        "tcc",
    "total_column_water_vapour":                "tcwv",
    "surface_solar_radiation_downwards":        "ssrd",
    "total_sky_direct_solar_radiation_at_surface": "fdir",
    "total_precipitation":                      "tp",
}

In [2]:
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
df["residual_24h"] = df["Congestion_SP15"] - df["Congestion_SP15"].shift(24)
df["large_deviation"] = (df["residual_24h"].abs() > df["residual_24h"].abs().quantile(0.75)).astype(int)


In [ ]:
# Profile the large deviation events
large_dev_idx = df[df["large_deviation"] == 1].index
small_dev_idx = df[df["large_deviation"] == 0].index

weather_cols = [c for c in df.columns if any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"])]
print("Weather during large deviation hours:")
print(df.loc[large_dev_idx, weather_cols + ["hour_of_day", "month"]].describe().round(2))
print("\nWeather during normal hours:")
print(df.loc[small_dev_idx, weather_cols + ["hour_of_day", "month"]].describe().round(2))

In [5]:
tscv = TimeSeriesSplit(n_splits=5, gap=48)
scaler = StandardScaler()
clf = HistGradientBoostingClassifier(max_iter=200, random_state=42)
lr = LogisticRegression(random_state=42, max_iter=500)

In [6]:
for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    print(f"Fold {fold}: train_end={df.index[train_idx[-1]]}, test_start={df.index[test_idx[0]]}")
    #assert df.index[train_idx[-1]] + pd.Timedelta(hours=24) < df.index[test_idx[0]]

Fold 0: train_end=3659, test_start=3708
Fold 1: train_end=7367, test_start=7416
Fold 2: train_end=11075, test_start=11124
Fold 3: train_end=14783, test_start=14832
Fold 4: train_end=18491, test_start=18540


In [7]:
def run_baselines(df, tscv, weather_cols, forecast=False):
    hgb_aucs = []
    lr_aucs = []
    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
        train_data = df.iloc[train_idx]
        # Compute threshold on TRAIN only
        if forecast:
            residual_24h_train = train_data["Congestion_NP15"].shift(-24) - train_data["Congestion_NP15"]
        else:
            # congestion[t] - congestion[t-24]
            residual_24h_train = train_data["Congestion_NP15"] - train_data["Congestion_NP15"].shift(24)
        threshold = residual_24h_train.abs().quantile(0.75)
        
        # Apply to test fold
        test_data = df.iloc[test_idx]
        if forecast:
            residual_24h_test = test_data["Congestion_NP15"].shift(-24) - test_data["Congestion_NP15"]
        else:
            residual_24h_test = test_data["Congestion_NP15"] - test_data["Congestion_NP15"].shift(24)
        y_test = (residual_24h_test.abs() > threshold).astype(int)
        
        X_test = test_data[weather_cols + ["hour_of_day", "month"]].dropna()
        y_test_clean = y_test.loc[X_test.index]
        
        if len(y_test_clean) < 10:  # Skip tiny folds
            print("Test set too small")
        
        # Train on this fold's train split, test on test split
        X_train = train_data[weather_cols + ["hour_of_day", "month"]].dropna()
        y_train = (residual_24h_train).abs().gt(threshold)
        y_train_clean = y_train.loc[X_train.index]
        
        # Inside your TimeSeriesSplit loop:
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        lr.fit(X_train_scaled, y_train_clean)
        auc_lr = roc_auc_score(y_test_clean, lr.predict_proba(X_test_scaled)[:, 1])
        
        clf.fit(X_train, y_train_clean)
        auc_hgb = roc_auc_score(y_test_clean, clf.predict_proba(X_test)[:, 1])
            
        hgb_aucs.append(auc_hgb)
        lr_aucs.append(auc_lr)
        
        print(f"Fold {fold}: threshold={threshold:.3f}, test N={len(y_test_clean)}, AUC={auc_hgb:.3f}")
        
        #residual_corr = train_data[weather_cols].corrwith(residual_24h_train)
        #congestion_corr = train_data[weather_cols].corrwith(y_train_clean)
        #print(f"Correlation with R24:\n {residual_corr}\n correlation with congestion:\n {congestion_corr}")
        #print(f"\nLeakage-corrected AUC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
    print(f"  Linear AUC:  {np.mean(lr_aucs):.3f}")
    print(f"  HGBoost AUC: {np.mean(hgb_aucs):.3f}")
    print(f"  Nonlinear gain: {np.mean(hgb_aucs) - np.mean(lr_aucs):.3f}")

In [13]:
weather_cols

['t2m_NP15',
 'ssrd_NP15',
 'fdir_NP15',
 't2m_SP15',
 'ssrd_SP15',
 'fdir_SP15',
 't2m_ZP26',
 'ssrd_ZP26',
 'fdir_ZP26',
 'wind_speed_NP15',
 'wind_speed_SP15',
 'wind_speed_ZP26']

In [8]:
# No weather columns
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
weather_cols = [c for c in df.columns if any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"])]
run_baselines(df, tscv, [], forecast=True)

Fold 0: threshold=1.380, test N=3708, AUC=0.730
Fold 1: threshold=2.831, test N=3708, AUC=0.814
Fold 2: threshold=2.789, test N=3708, AUC=0.783
Fold 3: threshold=2.825, test N=3708, AUC=0.804
Fold 4: threshold=2.914, test N=3708, AUC=0.834
  Linear AUC:  0.731
  HGBoost AUC: 0.793
  Nonlinear gain: 0.062


In [14]:
# Weather, no lag
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
weather_cols = [c for c in df.columns if any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"])]
run_baselines(df, tscv, weather_cols, forecast=True)

Fold 0: threshold=1.380, test N=3708, AUC=0.763
Fold 1: threshold=2.831, test N=3708, AUC=0.763
Fold 2: threshold=2.789, test N=3708, AUC=0.795
Fold 3: threshold=2.825, test N=3708, AUC=0.798
Fold 4: threshold=2.914, test N=3708, AUC=0.826
  Linear AUC:  0.777
  HGBoost AUC: 0.789
  Nonlinear gain: 0.012


In [15]:
# Temporal weather features
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
lags = [1, 6, 24]  # 1h persistence, 6h diurnal, 24h daily
weather_cols = [c for c in df.columns if any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"])]
# Lagged: stack weather lags
for lag in lags:
    for col in weather_cols:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)
weather_cols = [c for c in df.columns if ("lag" in c or any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"]))]
run_baselines(df, tscv, weather_cols, forecast=True)

Fold 0: threshold=1.380, test N=3708, AUC=0.738
Fold 1: threshold=2.831, test N=3708, AUC=0.782
Fold 2: threshold=2.789, test N=3708, AUC=0.808
Fold 3: threshold=2.825, test N=3708, AUC=0.796
Fold 4: threshold=2.914, test N=3708, AUC=0.822
  Linear AUC:  0.801
  HGBoost AUC: 0.789
  Nonlinear gain: -0.012


In [36]:
# Only use SP15 features
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
weather_cols = [c for c in df.columns if ("SP15" in c and any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed"]))]
run_baselines(df, tscv, weather_cols, forecast=True)

Fold 0: threshold=1.654, test N=3708, AUC=0.718
Fold 1: threshold=2.501, test N=3708, AUC=0.742
Fold 2: threshold=2.524, test N=3708, AUC=0.841
Fold 3: threshold=2.346, test N=3708, AUC=0.775
Fold 4: threshold=2.504, test N=3708, AUC=0.803
  Linear AUC:  0.729
  HGBoost AUC: 0.776
  Nonlinear gain: 0.046


In [20]:
import xarray as xr
np15_pt = xr.open_zarr("data/aligned/era5_spatial_NP15.zarr")
sp15_pt = xr.open_zarr("data/aligned/era5_spatial_SP15.zarr")

In [24]:
np15_ssrd = np15_pt

Data variables:
    d2m      (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    fdir     (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    sp       (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    ssrd     (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    t2m      (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    tcc      (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    tcwv     (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    tp       (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    u10      (time, latitude, longitude) float32 52MB dask.array<chunksize=(168, 23, 21), meta=np.ndarray>
    v10      (time, l

In [26]:
# Stratified AUC
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 
# Heat wave proxy (top 10% t2m_SP15)
df['heat_wave'] = df['t2m_SP15'] > df['t2m_SP15'].quantile(0.90)
df['north_south_solar_imbalance'] = (df['ssrd_NP15'] - df['ssrd_SP15']) * df['heat_wave']
weather_cols = [c for c in df.columns if any(v in c for v in ["ssrd", "fdir", "t2m", "wind_speed", "north_south_solar_imbalance"])]
heat_auc = run_baselines(df, tscv, weather_cols)

Fold 0: threshold=1.654, test N=3708, AUC=0.701
Fold 1: threshold=2.501, test N=3708, AUC=0.755
Fold 2: threshold=2.524, test N=3708, AUC=0.813
Fold 3: threshold=2.346, test N=3708, AUC=0.789
Fold 4: threshold=2.504, test N=3708, AUC=0.788
  Linear AUC:  0.764
  HGBoost AUC: 0.769
  Nonlinear gain: 0.005


In [ ]:
# Heat wave proxy (top 10% t2m_SP15)
df['heat_wave'] = df['t2m_SP15'] > df['t2m_SP15'].quantile(0.90)
df['north_south_solar_imbalance'] = (df['ssrd_NP15'] - df['ssrd_SP15']) * df['heat_wave']

# Stratified AUC
heat_auc = cv_auc(weather_cols + ['north_south_solar_imbalance'], 
                  df['large_deviation'], stratify=heat_wave)
print(f"Heat wave AUC gain: {heat_auc - baseline:.3f}")

In [65]:
df = pd.read_parquet("data/aligned/caiso_lmp_era5_averaged.parquet") 


In [68]:
df['time']

0       2023-01-09 08:00:00+00:00
1       2023-01-09 09:00:00+00:00
2       2023-01-09 10:00:00+00:00
3       2023-01-09 11:00:00+00:00
4       2023-01-09 12:00:00+00:00
                   ...           
22243   2026-01-01 03:00:00+00:00
22244   2026-01-01 04:00:00+00:00
22245   2026-01-01 05:00:00+00:00
22246   2026-01-01 06:00:00+00:00
22247   2026-01-01 07:00:00+00:00
Name: time, Length: 22248, dtype: datetime64[ns, UTC]